# some necessary libraries that might be needed compile and run

```bash
uv pip install "numpy<2"

uv pip install pandana
sudo apt install build-essential g++ python3-dev
sudo apt install gdal-bin libgdal-dev libgeos-dev libproj-dev
```

In [1]:
# ============================================================
# Who Can Reach What
# Urban Accessibility Analysis for Spatial Justice
# Münster Case Study
# <signed-off-by fillfeitosa@gmai.com>
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import pandana
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path



# --- Configuration ---
DATA_FILE        = "./data/data.geojson"
SOCIO_INDEX      = "pct_welfare_15_64"
DISTRICT_NAME    = "district_name"
POI_TYPE         = "bus_stop"
MAX_DISTANCE     = 1000   # meters — walking ~12 minutes
NUM_POIS         = 5      # count opportunities within MAX_DISTANCE
NETWORK_TYPE     = "walk"

# ox.settings.overpass_url = (
#     "https://overpass.kumi.systems/api/interpreter"
# )



ox.settings.overpass_url = "https://overpass-api.de/api/interpreter"

ox.settings.requests_timeout = 300
ox.settings.use_cache = True


Path("reports").mkdir(exist_ok=True)
print("Configuration ready")

Configuration ready


In [2]:
# ============================================================
# STEP 1 — Load and validate the GeoDataFrame
# ============================================================

def load_geodata(filename: str) -> gpd.GeoDataFrame:
    """
    Loads a GeoJSON file and reprojects to EPSG:25832
    (UTM zone 32N — metric CRS standard for Germany).
    
    EPSG:25832 is used for all spatial operations because
    it preserves metric distances and areas accurately.
    
    Args:
        filename: path to GeoJSON file
        
    Returns:
        GeoDataFrame in EPSG:25832
    """
    path = Path(filename)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    
    gdf = gpd.read_file(path)
    gdf = gdf.to_crs(epsg=25832)
    print(f"Loaded {len(gdf)} districts from {filename}")
    return gdf


gdf = load_geodata(DATA_FILE)
gdf[[DISTRICT_NAME, SOCIO_INDEX, "geometry"]].head()

Loaded 45 districts from ./data/data.geojson


,district_name,pct_welfare_15_64,geometry
0,Bahnhof,8.316961,"POLYGON ((406076.9 5756745.166, 406072.525 575..."
1,Albachten,4.996362,"POLYGON ((398729.006 5755247.927, 398738.761 5..."
2,Angelmodde,11.024479,"POLYGON ((409710.769 5751747.295, 409703.575 5..."
3,Kreuz,1.711358,"POLYGON ((405971.318 5758598.078, 405970.364 5..."
4,Berg Fidel,17.400530,"POLYGON ((405571.456 5752393.99, 405505.003 57..."


In [3]:
# ============================================================
# STEP 2 — Extract bounding box in WGS84
# ============================================================

def get_bbox_wgs84(gdf: gpd.GeoDataFrame) -> tuple:
    """
    Converts the GeoDataFrame extent to WGS84 and returns
    the bounding box as (north, south, east, west).
    
    osmnx requires lat/lon coordinates — we convert here
    once and reuse the result for all OSM downloads.
    The GeoDataFrame itself stays in EPSG:25832.
    
    Args:
        gdf: GeoDataFrame in any projected CRS
        
    Returns:
        (north, south, east, west) in decimal degrees
    """
    gdf_wgs84 = gdf.to_crs(epsg=4326)
    west, south, east, north = gdf_wgs84.total_bounds
    print(f"Bounding box — N:{north:.4f} S:{south:.4f} E:{east:.4f} W:{west:.4f}")
    return north, south, east, west


bbox = get_bbox_wgs84(gdf)

Bounding box — N:52.0600 S:51.8401 E:7.7744 W:7.4738


In [4]:
# ============================================================
# STEP 3 — Download OSM street network and build pandana graph
# ============================================================

def download_network(bbox: tuple, network_type: str = "walk"):
    """
    Downloads the OSM street network for the bounding box.
    
    Args:
        bbox:         (north, south, east, west) in WGS84
        network_type: 'walk', 'drive', or 'bike'
        
    Returns:
        osmnx MultiDiGraph
    """
    north, south, east, west = bbox
    print(f"Downloading OSM {network_type} network...")
    graph = ox.graph_from_bbox(
        bbox=(north, south, east, west),
        network_type=network_type
    )
    print(f"Network ready — {len(graph.nodes)} nodes, {len(graph.edges)} edges")
    return graph


def build_pandana_network(graph) -> pandana.Network:
    """
    Converts an osmnx graph to a pandana Network.
    Edge weights are lengths in meters as computed by osmnx.
    
    Args:
        graph: osmnx MultiDiGraph
        
    Returns:
        pandana.Network ready for accessibility queries
    """
    nodes, edges = ox.graph_to_gdfs(graph, nodes=True, edges=True)
    
    edge_from    = pd.Series(edges.index.get_level_values("u").values)
    edge_to      = pd.Series(edges.index.get_level_values("v").values)
    edge_weights = pd.DataFrame({"distance": edges["length"].values})
    
    network = pandana.Network(
        node_x=nodes["x"],
        node_y=nodes["y"],
        edge_from=edge_from,
        edge_to=edge_to,
        edge_weights=edge_weights,
        twoway=False
    )
    print(f"Pandana network ready — {len(nodes)} nodes")
    return network


# graph   = download_network(bbox, network_type=NETWORK_TYPE)
graph = ox.graph_from_place(
    "Münster, Germany",
    network_type="bike"
)
# graph = ox.graph_from_("./muenster-regbez-260525.osm.pbf")
network = build_pandana_network(graph)

SSLError: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter/interpreter (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))

In [ ]:
ox.save_graphml(graph, "muenster_walk.graphml")

In [ ]:
# ============================================================
# STEP 4 — Download POIs and register on pandana network
# ============================================================

POI_TAGS = {
    "school":            {"amenity": "school"},
    "kindergarten":      {"amenity": "kindergarten"},
    "bus_stop":          {"highway": "bus_stop"},
    "hospital":          {"amenity": "hospital"},
    "clinic":            {"amenity": "clinic"},
    "pharmacy":          {"amenity": "pharmacy"},
    "supermarket":       {"shop": "supermarket"},
    "park":              {"leisure": "park"},
    "employment_center": {"office": "employment_agency"},
}


def download_pois(bbox: tuple, poi_type: str) -> gpd.GeoDataFrame:
    """
    Downloads POIs from OSM for the given category.
    Polygon geometries (e.g. school buildings) are
    converted to centroids for pandana compatibility.
    
    Args:
        bbox:     (north, south, east, west) in WGS84
        poi_type: key from POI_TAGS dictionary
        
    Returns:
        GeoDataFrame of POI point geometries in WGS84
    """
    if poi_type not in POI_TAGS:
        raise ValueError(
            f"Unknown POI type '{poi_type}'. "
            f"Available: {list(POI_TAGS.keys())}"
        )
    
    north, south, east, west = bbox
    tags = POI_TAGS[poi_type]
    
    print(f"Downloading {poi_type} POIs from OSM...")
    pois = ox.features_from_bbox(
        bbox=(north, south, east, west),
        tags=tags
    )
    pois = pois.to_crs(epsg=4326)
    pois["geometry"] = pois.geometry.centroid
    print(f"Found {len(pois)} {poi_type} locations")
    return pois


def register_pois(
    network:      pandana.Network,
    pois:         gpd.GeoDataFrame,
    poi_type:     str,
    max_distance: float = MAX_DISTANCE,
    max_items:    int   = NUM_POIS,
) -> None:
    """
    Registers POI locations on the pandana network.
    Must be called before compute_accessibility.
    
    max_distance and max_items here must match what
    is later passed to nearest_pois — they define the
    search space pandana prepares internally.
    
    Args:
        network:      pandana Network object
        pois:         GeoDataFrame of POI points in WGS84
        poi_type:     category name matching POI_TAGS
        max_distance: maximum search radius in meters
        max_items:    maximum number of POIs to return per node
    """
    network.set_pois(
        category=poi_type,
        maxdist=max_distance,
        maxitems=max_items,
        x_col=pois.geometry.x,
        y_col=pois.geometry.y,
    )
    print(f"{poi_type} POIs registered on network")


pois = download_pois(bbox, POI_TYPE)
register_pois(network, pois, POI_TYPE)
pois.head(3)